# the code below will get the annotation data and align with ACC data

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ACC_PATH = Path("../../src/test_fernando_align/AlgorithmData-Acceleration-[20251011-20251011]-1113_SHIFTED_Z.csv")
ANN_PATH = Path("../../src/test_fernando_align/20251011_111215_annotations_gopro.csv")

OUT_SNIPPET_PATH = Path("../../src/test_fernando_align/acc_labeled_testwindow.csv")
OUT_FULL_PATH    = Path("../../src/test_fernando_align/acc_with_behavior_full.csv")

ACC_TIME_COL = "Collecting time"
ANN_START_COL = "start_datetime"
ANN_END_COL = "end_datetime"
ANN_BEHAVIOR_COL = "behavior"
ANN_VIDEO_COL = "video_filename"

CHUNKSIZE = 250000

# True = force annotation date to match video filename date


FIX_ANNOTATION_DATE_FROM_VIDEO_FILENAME = True


def to_utc_mixed(series):
    return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")


def extract_video_start_from_filename(name):
    """
    Example:
    20251011_111215.MP4 -> 2025-10-11 11:12:15 UTC
    """
    if pd.isna(name):
        return pd.NaT

    name = str(name)
    m = re.search(r"(\d{8})_(\d{6})", name)
    if not m:
        return pd.NaT

    dt_str = m.group(1) + m.group(2)  # YYYYMMDDHHMMSS
    return pd.to_datetime(dt_str, format="%Y%m%d%H%M%S", utc=True)


def fix_annotation_dates_from_video_filename(ann):
    if ANN_VIDEO_COL not in ann.columns:
        print("Warning: video_filename column not found. Cannot auto-fix annotation date.")
        return ann

    video_start = extract_video_start_from_filename(ann[ANN_VIDEO_COL].iloc[0])
    if pd.isna(video_start):
        print("Warning: could not extract datetime from video filename. Cannot auto-fix annotation date.")
        return ann

    # Use relative times in seconds from the annotation file
    if "start_time_s" not in ann.columns or "end_time_s" not in ann.columns:
        print("Warning: start_time_s/end_time_s columns not found. Cannot auto-fix annotation date.")
        return ann

    ann[ANN_START_COL] = video_start + pd.to_timedelta(ann["start_time_s"], unit="s")
    ann[ANN_END_COL]   = video_start + pd.to_timedelta(ann["end_time_s"], unit="s")

    print(f"Annotation timestamps rebuilt from video filename start: {video_start}")
    return ann



ann = pd.read_csv(ANN_PATH)

required_ann_cols = [ANN_START_COL, ANN_END_COL, ANN_BEHAVIOR_COL]
for col in required_ann_cols:
    if col not in ann.columns:
        raise ValueError(f"Column '{col}' not found in annotation file. Found: {list(ann.columns)}")


ann[ANN_START_COL] = to_utc_mixed(ann[ANN_START_COL])
ann[ANN_END_COL]   = to_utc_mixed(ann[ANN_END_COL])

print("Raw annotation rows:", len(ann))
print("Parsed start datetimes:", ann[ANN_START_COL].notna().sum())
print("Parsed end datetimes:", ann[ANN_END_COL].notna().sum())


if FIX_ANNOTATION_DATE_FROM_VIDEO_FILENAME:
    ann = fix_annotation_dates_from_video_filename(ann)

ann = ann.dropna(subset=[ANN_START_COL, ANN_END_COL, ANN_BEHAVIOR_COL]).copy()
ann = ann.sort_values(ANN_START_COL).reset_index(drop=True)

print("Loaded annotations after cleaning:", len(ann))
if len(ann) == 0:
    raise ValueError("No valid annotations found after parsing/fixing datetimes.")

window_start = ann[ANN_START_COL].min()
window_end   = ann[ANN_END_COL].max()

print("Annotation window:", window_start, "to", window_end)


ann_start_ns = ann[ANN_START_COL].astype("int64").to_numpy()
ann_end_ns   = ann[ANN_END_COL].astype("int64").to_numpy()
ann_beh      = ann[ANN_BEHAVIOR_COL].astype(str).to_numpy()

# Check ACC range first$%$%%%%%%%%%%%%%%%%%%

acc_preview = pd.read_csv(ACC_PATH, usecols=[ACC_TIME_COL])
acc_preview[ACC_TIME_COL] = to_utc_mixed(acc_preview[ACC_TIME_COL])
acc_preview = acc_preview.dropna(subset=[ACC_TIME_COL])

if len(acc_preview) == 0:
    raise ValueError("No valid accelerometer timestamps found.")

acc_min = acc_preview[ACC_TIME_COL].min()
acc_max = acc_preview[ACC_TIME_COL].max()

print("ACC window:", acc_min, "to", acc_max)

overlap_start = max(acc_min, window_start)
overlap_end   = min(acc_max, window_end)

if overlap_start > overlap_end:
    print("\nWARNING: No temporal overlap between ACC and annotation data.")
    print("This usually means the annotation date/time is wrong or shifted incorrectly.")
    print("ACC range:", acc_min, "to", acc_max)
    print("ANN range:", window_start, "to", window_end)
else:
    print("\nOverlap detected:", overlap_start, "to", overlap_end)


first_snip = True
first_full = True
rows_written_snip = 0
rows_written_full = 0
rows_labeled = 0

for chunk in pd.read_csv(ACC_PATH, chunksize=CHUNKSIZE):
    if ACC_TIME_COL not in chunk.columns:
        raise ValueError(f"Column '{ACC_TIME_COL}' not found in accelerometer file. Found: {list(chunk.columns)}")

    chunk[ACC_TIME_COL] = to_utc_mixed(chunk[ACC_TIME_COL])
    chunk = chunk.dropna(subset=[ACC_TIME_COL]).copy()

    if len(chunk) == 0:
        continue

    t_ns = chunk[ACC_TIME_COL].astype("int64").to_numpy()
    labels_full = np.full(len(chunk), np.nan, dtype=object)

    for i in range(len(ann)):
        mask = (t_ns >= ann_start_ns[i]) & (t_ns <= ann_end_ns[i])
        if mask.any():
            labels_full[mask] = ann_beh[i]

    labeled_mask = pd.notna(labels_full)
    rows_labeled += int(labeled_mask.sum())

    # Save full output
    chunk_full = chunk.copy()
    chunk_full["behavior_label"] = labels_full
    chunk_full.to_csv(
        OUT_FULL_PATH,
        index=False,
        mode="w" if first_full else "a",
        header=first_full
    )
    first_full = False
    rows_written_full += len(chunk_full)

    # Save snippet output only for annotation time window
    in_window = (chunk[ACC_TIME_COL] >= window_start) & (chunk[ACC_TIME_COL] <= window_end)
    if in_window.any():
        chunk_snip = chunk.loc[in_window].copy()
        chunk_snip["behavior_label"] = np.asarray(labels_full, dtype=object)[in_window.to_numpy()]
        chunk_snip.to_csv(
            OUT_SNIPPET_PATH,
            index=False,
            mode="w" if first_snip else "a",
            header=first_snip
        )
        first_snip = False
        rows_written_snip += len(chunk_snip)

print("\nDone. 🦇")
print("Snippet rows written:", rows_written_snip)
print("Snippet output saved to:", OUT_SNIPPET_PATH)
print("Full rows written:", rows_written_full)
print("Full output saved to:", OUT_FULL_PATH)
print("Rows that received a behavior label:", rows_labeled)